# Data Preprocessing & Cleaning (`Data-Preprocessing/Preprocessing-and-Cleaning.ipynb`)
**Progree Data Analytics Internship**

### Overview
This notebook implements a structured **Data Preprocessing and Cleaning Pipeline** on a raw retail/e-commerce dataset—covering data inspection, missing value diagnosis & imputation, duplicate removal, text standardization, domain rule validation, IQR outlier treatment, and feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)

## 1. Load Raw Data & Inspect Structure

In [ ]:
df_raw = pd.read_csv('raw_retail_dataset.csv')
print('Initial Shape:', df_raw.shape)
display(df_raw.head(10))

print('\nData Types:')
print(df_raw.dtypes)

print('\nMissing Values Count:')
print(df_raw.isnull().sum())

print('\nDuplicate Rows Count:', df_raw.duplicated().sum())

## 2. Remove Duplicates & Standardize Categorical Fields

In [ ]:
df = df_raw.copy()

# Drop exact duplicate rows
before_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before_rows - len(df)} duplicate rows. Remaining rows: {len(df)}')

# Standardize casing and whitespace in Category & City
df['Category'] = df['Category'].astype(str).str.strip().str.title()
df['City'] = df['City'].astype(str).replace('nan', np.nan).str.strip().str.title()

print('\nCleaned Categories:', df['Category'].unique())
print('Missing Cities before imputation:', df['City'].isnull().sum())

## 3. Domain Validation & Systematic Missing Value Imputation

In [ ]:
# Convert invalid domain values (negative or unrealistic numbers) to NaN before imputation
df.loc[(df['Customer_Age'] < 18) | (df['Customer_Age'] > 100), 'Customer_Age'] = np.nan
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df.loc[df['Unit_Price_PKR'] <= 0, 'Unit_Price_PKR'] = np.nan

# Impute City using Mode
df['City'] = df['City'].fillna(df['City'].mode()[0])

# Impute Customer_Age and Quantity using overall Median
df['Customer_Age'] = df['Customer_Age'].fillna(df['Customer_Age'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# Impute Unit_Price_PKR using Category-wise Median
df['Unit_Price_PKR'] = df.groupby('Category')['Unit_Price_PKR'].transform(
    lambda x: x.fillna(x.median())
)

# Impute missing Discount_Pct as 0% (no discount applied)
df['Discount_Pct'] = df['Discount_Pct'].fillna(0.0)

print('Missing Values After Imputation:')
print(df.isnull().sum())

## 4. Outlier Detection & IQR Capping

In [ ]:
def iqr_cap(series, lower_floor=0.0):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = max(lower_floor, q1 - 1.5 * iqr)
    upper_bound = q3 + 1.5 * iqr
    return series.clip(lower=lower_bound, upper=upper_bound)

# Visualize before & after outlier capping for Unit_Price_PKR
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=df['Unit_Price_PKR'], ax=axes[0], color='#2C5E7A')
axes[0].set_title('Unit_Price_PKR Before IQR Capping')

df['Unit_Price_PKR'] = iqr_cap(df['Unit_Price_PKR'], lower_floor=0.0)
df['Quantity'] = iqr_cap(df['Quantity'], lower_floor=1.0)

sns.boxplot(x=df['Unit_Price_PKR'], ax=axes[1], color='#1B365D')
axes[1].set_title('Unit_Price_PKR After IQR Capping')

plt.tight_layout()
plt.show()

## 5. Feature Engineering & Export Preprocessed Dataset

In [ ]:
# Cast numeric columns to clean types
df['Customer_Age'] = df['Customer_Age'].astype(int)
df['Quantity'] = df['Quantity'].astype(int)
df['Unit_Price_PKR'] = np.round(df['Unit_Price_PKR'], 2)

# Compute Gross & Net Revenue
df['Gross_Revenue_PKR'] = np.round(df['Quantity'] * df['Unit_Price_PKR'], 2)
df['Net_Revenue_PKR'] = np.round(df['Gross_Revenue_PKR'] * (1 - df['Discount_Pct'] / 100.0), 2)

display(df.head())
display(df.describe())

df.to_csv('preprocessed_data.csv', index=False)
print('Saved preprocessed_data.csv successfully. Final shape:', df.shape)